In [40]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType
import pyarrow as pa
from pathlib import Path
import pyarrow.parquet as pq
from scipy.signal import find_peaks
from scipy.ndimage import label
import plotly.graph_objects as go


In [4]:
with uproot.open(r"data\raw\Run2012BC_DoubleMuParked_Muons.root") as f:
    print(f.keys())
    tree = f["Events"]
    print(f"Nombre d'événements : {tree.num_entries:,}")
    for name, branch in tree.items():
        print(f"  {name:<25} {branch.typename}")

['Events;75', 'Events;74']
Nombre d'événements : 61,540,413
  nMuon                     uint32_t
  Muon_pt                   float[]
  Muon_eta                  float[]
  Muon_phi                  float[]
  Muon_mass                 float[]
  Muon_charge               int32_t[]


Le **LHC** (Large Hadron Collider) au CERN est le plus grand accélérateur de particules du monde.
Concrètement : deux faisceaux de protons sont lancés en sens inverse dans un tunnel de 27 km,
puis percutés l'un contre l'autre à une vitesse proche de celle de la lumière.

À chaque collision, l'énergie libérée crée de nouvelles particules — comme `E = mc²` en sens inverse :
de l'énergie qui se transforme en matière.

**CMS** (Compact Muon Solenoid) est l'un des 4 détecteurs géants installés autour du LHC.
Son rôle : photographier ce qui sort de chaque collision, des millions de fois par seconde.

Ce dataset contient **61.5 millions de collisions** enregistrées en 2012, filtrées pour ne garder
que celles où CMS a détecté au moins un **muon**.

---

## C'est quoi un muon ?

Un muon, c'est comme un électron — mais environ **200 fois plus lourd**.

Ce qui le rend utile en physique des particules :
- Il **traverse** le détecteur sans s'arrêter (contrairement aux électrons ou aux photons)
- Il est **facile à détecter** avec précision
- Il est produit lors de la désintégration de nombreuses particules intéressantes

---

## Les 6 variables du dataset

Chaque ligne du dataset représente **un événement** (= une collision).
Pour chaque événement, on a :

### `nMuon` — Combien de muons ont été détectés ?
Un entier : 0, 1, 2, 3...

---

### `Muon_pt` — À quelle vitesse le muon se déplace-t-il ? *(en GeV (Giga ElectronVolt): unité d'énergie où 1GeV ~ 1,6.10^-10J)*

**pt** = *transverse momentum* = impulsion transverse.
- *Elle est la composante de sa quantité de mouvement perpendiculaire à l'axe du faisceau dans un collisionneur*

- Valeur typique : 5 à 50 GeV
- Elle sert de filtre pour s'intéresser aux colissions intéressantes et éliminer les bruits de fond
---

### `Muon_eta` — Dans quelle direction verticale part le muon ? 

**eta** = *pseudorapidité* = angle par rapport à l'axe du faisceau.

- Plage dans CMS : **-2.4 à +2.4**
- Au-delà de ±2.4, le détecteur ne voit plus rien → **zone morte**

---

### `Muon_phi` — Dans quelle direction horizontale part le muon ? *(en radians)*

**phi** = rotation autour de l'axe du faisceau.

- Plage : **-π à +π** 
- Combiné à `eta`, il donne la **direction complète** dans l'espace

---

### `Muon_mass` — Quelle est la masse du muon ? *(en GeV)*

La masse du muon est une **constante de la nature** : ~0.106 GeV.
Elle est identique pour tous les muons, tout le temps, partout dans l'univers.

---

### `Muon_charge` — Le muon est-il positif ou négatif ?

Valeur : **+1** ou **-1**.

Une particule **neutre** qui se désintègre en deux muons produit **nécessairement**
un muon+ et un muon−. Leur somme de charges = 0.

---

## Étapes à suivre:
- Analyser la distribution statistique des variables
- Filter les valeurs moins intéressantes
- On va charger 500k évènements, ce qui donnera déjà une bonne explication statistique


In [5]:
with uproot.open(r"data\raw\Run2012BC_DoubleMuParked_Muons.root") as f:
    tree = f["Events"]
    arrays = tree.arrays(
        ["nMuon", "Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"],
        entry_stop=500_000,
        library="ak"
    )

# ak.flatten() "aplatit" les arrays :
# avant : [[10.7, 15.7], [10.5], [3.2, ...], ...]  ← un array par événement
# après : [10.7, 15.7, 10.5, 3.2, ...]             ← un muon par ligne
pt     = ak.to_numpy(ak.flatten(arrays["Muon_pt"]))
eta    = ak.to_numpy(ak.flatten(arrays["Muon_eta"]))
phi    = ak.to_numpy(ak.flatten(arrays["Muon_phi"]))
mass   = ak.to_numpy(ak.flatten(arrays["Muon_mass"]))
charge = ak.to_numpy(ak.flatten(arrays["Muon_charge"]))
nmuon  = ak.to_numpy(arrays["nMuon"])

print(f"Événements chargés : {len(nmuon):,}")
print(f"Muons total        : {len(pt):,}")
print(f"Muons par événement (moyenne) : {len(pt)/len(nmuon):.2f}")

Événements chargés : 500,000
Muons total        : 1,179,523
Muons par événement (moyenne) : 2.36


In [6]:
unique, counts = np.unique(nmuon, return_counts=True)

print("Distribution nMuon :")
print(f"{'nMuon':<10} {'Événements':>12} {'Pourcentage':>12}")
print("-" * 36)
for u, c in zip(unique, counts):
    print(f"{u:<10} {c:>12,} {c/len(nmuon)*100:>11.1f}%")

print(f"\nTotal événements : {len(nmuon):,}")
print(f"Événements avec exactement 2 muons : {counts[unique==2][0]:,} ({counts[unique==2][0]/len(nmuon)*100:.1f}%)")

Distribution nMuon :
nMuon        Événements  Pourcentage
------------------------------------
0                15,677         3.1%
1                67,070        13.4%
2               244,947        49.0%
3               108,368        21.7%
4                40,199         8.0%
5                14,183         2.8%
6                 5,395         1.1%
7                 2,193         0.4%
8                 1,003         0.2%
9                   448         0.1%
10                  237         0.0%
11                  119         0.0%
12                   54         0.0%
13                   38         0.0%
14                   32         0.0%
15                   14         0.0%
16                    6         0.0%
17                    4         0.0%
18                    2         0.0%
20                    1         0.0%
22                    2         0.0%
23                    2         0.0%
24                    1         0.0%
25                    1         0.0%
28               

## Analyse — Distribution du nombre de muons par événement

| nMuon | Événements | % |
|-------|-----------|---|
| 0 | 15 677 | 3.1% |
| 1 | 67 070 | 13.4% |
| 2 | 244 947 | 49.0% |
| 3 | 108 368 | 21.7% |
| 4+ | 63 492 | 12.7% |

### Interprétation

**0 muon (3.1%)** : aucun muon reconstruit dans l'événement.
Non exploitable — pas de donnée muon disponible.

**1 muon (13.4%)** : un seul muon détecté.
Exploitable pour des analyses sur les muons individuels
(distribution de pt, eta, phi d'un muon isolé).
Non exploitable pour toute analyse nécessitant une paire.

**2 muons (49%)** : deux muons détectés.
Exploitable pour toute analyse par paires — une seule combinaison possible,
pas d'ambiguïté sur l'appariement.

**3 muons (21.7%)** : trois muons détectés.
Exploitable pour des analyses spécifiques aux événements multi-muons.
Pour une analyse par paires, il existe 3 combinaisons possibles — (A-B, A-C ou B-C)
un choix de sélection est donc nécessaire

**4 muons et plus (12.7%)** : quatre muons ou plus détectés.
Potentiellement 2 paires indépendantes
Ou alors une paire de muons et 2 muons indépendants


---

### Ce que ces chiffres disent du dataset

La majorité des événements (49%) contient exactement 2 muons —

Les événements à 3 muons et plus (35%) reflètent le **pileup** :
plusieurs collisions simultanées lors du même croisement de faisceaux

### Conclusion

Pour des raisons de simplicité, nous nous intéresserons aux évènements produisant des di-muons que nous savons être une paire

In [7]:
print("Stats descriptives — pt (GeV) :")
print(f"  Minimum   : {pt.min():.3f} GeV")
print(f"  Maximum   : {pt.max():.1f} GeV")
print(f"  Moyenne   : {pt.mean():.3f} GeV")
print(f"  Médiane   : {np.median(pt):.3f} GeV")
print(f"  Std       : {pt.std():.3f} GeV")

print("\nPercentiles pt :")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  p{p:<5} : {np.percentile(pt, p):.2f} GeV")

print("\nFraction de muons selon seuil pt :")
for threshold in [1, 2, 3, 5, 10]:
    frac = np.sum(pt < threshold) / len(pt) * 100
    print(f"  pt < {threshold} GeV : {frac:.1f}% des muons éliminés")

Stats descriptives — pt (GeV) :
  Minimum   : 3.000 GeV
  Maximum   : 157191.2 GeV
  Moyenne   : 17.256 GeV
  Médiane   : 11.568 GeV
  Std       : 202.390 GeV

Percentiles pt :
  p50    : 11.57 GeV
  p75    : 18.95 GeV
  p90    : 32.59 GeV
  p95    : 42.69 GeV
  p99    : 67.06 GeV
  p99.9  : 286.74 GeV

Fraction de muons selon seuil pt :
  pt < 1 GeV : 0.0% des muons éliminés
  pt < 2 GeV : 0.0% des muons éliminés
  pt < 3 GeV : 0.0% des muons éliminés
  pt < 5 GeV : 15.8% des muons éliminés
  pt < 10 GeV : 42.8% des muons éliminés


In [8]:
print("Analyse d'outliers — pt :")
for threshold in [10, 20 ,30, 40, 50, 100, 500, 1000, 10000]:
    n = np.sum(pt > threshold)
    print(f"  pt > {threshold:<6} GeV : {n:>6,} muons ({n/len(pt)*100:.4f}%)")
print("--------------------------------------------")
print(f"Muons au-dessus : {np.sum(pt > 100):,} ({np.sum(pt > 100)/len(pt)*100:.4f}%)")
print(f"\nSans les outliers (pt < 100 GeV) :")
pt_clean = pt[pt < 100]
print(f"  Moyenne : {pt_clean.mean():.3f} GeV")
print(f"  Std     : {pt_clean.std():.3f} GeV")

Analyse d'outliers — pt :
  pt > 10     GeV : 674,643 muons (57.1963%)
  pt > 20     GeV : 269,568 muons (22.8540%)
  pt > 30     GeV : 136,293 muons (11.5549%)
  pt > 40     GeV : 74,110 muons (6.2830%)
  pt > 50     GeV : 29,273 muons (2.4818%)
  pt > 100    GeV :  4,833 muons (0.4097%)
  pt > 500    GeV :    716 muons (0.0607%)
  pt > 1000   GeV :    397 muons (0.0337%)
  pt > 10000  GeV :     25 muons (0.0021%)
--------------------------------------------
Muons au-dessus : 4,833 (0.4097%)

Sans les outliers (pt < 100 GeV) :
  Moyenne : 15.195 GeV
  Std     : 12.366 GeV


## Analyse — Variable pt (impulsion transverse)

### Observations

**Minimum exactement à 3.0 GeV** :
Le dataset est donc déjà pré-filtré côté bas : aucun minorant à appliquer.

**Outliers > 100 GeV** : 4 833 muons (0.41%) dépassent 100 GeV.
Ces valeurs sont vraisemblablement des outliers

| Seuil | Muons au-dessus | % du total |
|-------|----------------|------------|
| 10 GeV | 674,643 | 57.1963% |
| 50 GeV | 269,568 | 22.8540% |
| 100 GeV | 4 833 | 0.41% |
| 1 000 GeV | 397 | 0.03% |

### Décision

On conserve uniquement les muons avec **pt < 100 GeV**.

Cela représente **99.6% des données** — aucune perte significative de signal.
La distribution devient cohérente : moyenne ~15 GeV, std ~12 GeV.

Le filtre final sur pt est donc : **3 GeV ≤ pt < 100 GeV**

In [9]:
print("Stats descriptives — eta :")
print(f"  Minimum : {eta.min():.3f}")
print(f"  Maximum : {eta.max():.3f}")
print(f"  Moyenne : {eta.mean():.3f}")
print(f"  Médiane : {np.median(eta):.3f}")
print(f"  Std     : {eta.std():.3f}")

print("\nDistribution par zone :")
zones = [
    ("Zone morte négative  (eta < -2.4)", eta < -2.4),
    ("Endcap négatif (-2.4 à -1.6)"     , (eta >= -2.4) & (eta < -1.6)),
    ("Barrel          (-1.6 à +1.6)"    , (eta >= -1.6) & (eta <= 1.6)),
    ("Endcap positif  (+1.6 à +2.4)"    , (eta > 1.6)  & (eta <= 2.4)),
    ("Zone morte positive  (eta > +2.4)", eta > 2.4),
]
for label, mask in zones:
    n = np.sum(mask)
    print(f"  {label} : {n:>7,} ({n/len(eta)*100:.1f}%)")

Stats descriptives — eta :
  Minimum : -6.575
  Maximum : 9.117
  Moyenne : 0.017
  Médiane : 0.012
  Std     : 1.292

Distribution par zone :
  Zone morte négative  (eta < -2.4) :   4,647 (0.4%)
  Endcap négatif (-2.4 à -1.6) : 150,307 (12.7%)
  Barrel          (-1.6 à +1.6) : 859,455 (72.9%)
  Endcap positif  (+1.6 à +2.4) : 160,079 (13.6%)
  Zone morte positive  (eta > +2.4) :   5,035 (0.4%)


In [10]:
print("Analyse zones mortes — eta :")
eta_clean = eta[(eta >= -2.4) & (eta <= 2.4)]
print(f"  Muons dans l'acceptance (-2.4 à +2.4) : {len(eta_clean):,} ({len(eta_clean)/len(eta)*100:.1f}%)")
print(f"  Muons hors acceptance                 : {len(eta)-len(eta_clean):,} ({(len(eta)-len(eta_clean))/len(eta)*100:.1f}%)")

print("\nSymétrie barrel :")
barrel_neg = np.sum((eta >= -1.6) & (eta < 0))
barrel_pos = np.sum((eta >= 0) & (eta <= 1.6))
print(f"  Barrel négatif (eta -1.6 à 0)  : {barrel_neg:,} ({barrel_neg/len(eta)*100:.1f}%)")
print(f"  Barrel positif (eta  0 à +1.6) : {barrel_pos:,} ({barrel_pos/len(eta)*100:.1f}%)")

print("\nSymétrie endcaps :")
print(f"  Endcap négatif : {np.sum((eta >= -2.4) & (eta < -1.6)):,}")
print(f"  Endcap positif : {np.sum((eta > 1.6) & (eta <= 2.4)):,}")

Analyse zones mortes — eta :
  Muons dans l'acceptance (-2.4 à +2.4) : 1,169,841 (99.2%)
  Muons hors acceptance                 : 9,682 (0.8%)

Symétrie barrel :
  Barrel négatif (eta -1.6 à 0)  : 429,666 (36.4%)
  Barrel positif (eta  0 à +1.6) : 429,789 (36.4%)

Symétrie endcaps :
  Endcap négatif : 150,307
  Endcap positif : 160,079


In [11]:
print("Stats descriptives — phi (radians) :")
print(f"  Minimum : {phi.min():.3f}")
print(f"  Maximum : {phi.max():.3f}")
print(f"  Moyenne : {phi.mean():.3f}")
print(f"  Médiane : {np.median(phi):.3f}")
print(f"  Std     : {phi.std():.3f}")

print("\nDistribution par quart :")
quarts = [
    ("-π  à -π/2", (phi >= -np.pi)   & (phi < -np.pi/2)),
    ("-π/2 à  0 ", (phi >= -np.pi/2) & (phi < 0)),
    (" 0  à +π/2", (phi >= 0)        & (phi < np.pi/2)),
    ("+π/2 à +π ", (phi >= np.pi/2)  & (phi <= np.pi)),
]
for label, mask in quarts:
    n = np.sum(mask)
    print(f"  {label} : {n:>7,} ({n/len(phi)*100:.1f}%)")

Stats descriptives — phi (radians) :
  Minimum : -3.142
  Maximum : 3.142
  Moyenne : 0.019
  Médiane : 0.021
  Std     : 1.818

Distribution par quart :
  -π  à -π/2 : 293,447 (24.9%)
  -π/2 à  0  : 292,679 (24.8%)
   0  à +π/2 : 292,647 (24.8%)
  +π/2 à +π  : 300,750 (25.5%)


## Analyse — Variables eta et phi

### eta (pseudorapidité) — position angulaire du muon

L'eta mesure l'angle du muon par rapport à l'axe du faisceau.
- `eta = 0` : le muon part perpendiculairement au faisceau
- `|eta| = 2.4` : le muon part presque dans la direction du faisceau

**Pourquoi ±2.4 comme limite ?**
C'est la limite physique du détecteur CMS.
Au-delà de ±2.4, il n'y a plus de matériel de détection —
les muons qui partent trop "vers l'avant" ne sont tout simplement pas captés.

**Résultats observés :**

| Zone | eta | Muons | % |
|------|-----|-------|---|
| Zone morte négative | < -2.4 | 4 647 | 0.4% |
| Endcap négatif | -2.4 à -1.6 | 150 307 | 12.7% |
| Barrel central | -1.6 à +1.6 | 859 455 | 72.9% |
| Endcap positif | +1.6 à +2.4 | 160 079 | 13.6% |
| Zone morte positive | > +2.4 | 5 035 | 0.4% |

**Barrel vs Endcaps** : le détecteur CMS est composé de deux parties.
Le **barrel** est la partie cylindrique centrale.
Les **endcaps** sont les deux "bouchons" aux extrémités —
ils couvrent les angles plus rasants.

**Symétrie parfaite du barrel** : 36.4% de chaque côté.
C'est une validation que le détecteur fonctionne correctement
et que les données sont fiables.

**Légère asymétrie des endcaps** : 150k vs 160k (~6% d'écart).

**Outliers eta** : 9 682 muons (0.8%) ont une valeur eta hors de ±2.4,
certains atteignant -6.5 ou +9.1 — valeurs physiquement impossibles.

**Filtre appliqué : -2.4 ≤ eta ≤ +2.4** → conserve 99.2% des muons.

---

### phi (angle azimutal) — rotation autour du faisceau

Le phi mesure la direction du muon autour de l'axe du faisceau.
 Il va de -π à +π (soit -3.14 à +3.14 radians), comme un cercle trigonométrique.

**Résultats observés :**

| Quart | phi | Muons | % |
|-------|-----|-------|---|
| -π à -π/2 | | 293 447 | 24.9% |
| -π/2 à 0 | | 292 679 | 24.8% |
| 0 à +π/2 | | 292 647 | 24.8% |
| +π/2 à +π | | 300 750 | 25.5% |

**Distribution quasi-uniforme** : ~25% dans chaque quart.
Aucune direction ne semble favorisée

**Pas d'outliers** : min -3.142, max +3.142 — exactement les bornes naturelles.
Aucun filtre nécessaire sur phi.

---

### Récapitulatif des filtres retenus

| Variable | Filtre | 
|----------|--------|
| pt | 3 ≤ pt < 100 GeV |
| eta | -2.4 ≤ eta ≤ +2.4 |
| phi | aucun |

In [12]:
print("Stats descriptives — mass (GeV) :")
print(f"  Minimum : {mass.min():.5f} GeV")
print(f"  Maximum : {mass.max():.5f} GeV")
print(f"  Moyenne : {mass.mean():.5f} GeV")
print(f"  Médiane : {np.median(mass):.5f} GeV")
print(f"  Std     : {mass.std():.6f} GeV")
print(f"  Valeur PDG officielle : 0.10566 GeV")

print("\nStats descriptives — charge :")
unique_q, counts_q = np.unique(charge, return_counts=True)
for q, c in zip(unique_q, counts_q):
    print(f"  charge {int(q):+d} : {c:>7,} ({c/len(charge)*100:.1f}%)")
print(f"\n  Ratio +/- : {counts_q[1]/counts_q[0]:.4f}")

Stats descriptives — mass (GeV) :
  Minimum : 0.10561 GeV
  Maximum : 0.10567 GeV
  Moyenne : 0.10566 GeV
  Médiane : 0.10566 GeV
  Std     : 0.000000 GeV
  Valeur PDG officielle : 0.10566 GeV

Stats descriptives — charge :
  charge -1 : 582,364 (49.4%)
  charge +1 : 597,159 (50.6%)

  Ratio +/- : 1.0254


# RÉCAPITULATIF EDA — Décisions de filtrage

---

- **nMuon** : garder `nMuon == 2` uniquement  
  → 244 947 événements (49,0 %)

- **pt** : `3 GeV ≤ pt < 100 GeV`  
  → élimine 4 833 muons (0,41 %)

- **eta** : `-2.4 ≤ eta ≤ +2.4`  
  → élimine 9 682 muons (0,82 %)

- **phi** : aucun filtre

- **mass** : constante à `0.10566 GeV` — ignorée

- **charge** : filtre charge opposée appliqué lors du calcul di‑muon

---

In [13]:
PARQUET_DIR = Path("data/processed/dimuon_parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 5_000_000

with uproot.open(r"data\raw\Run2012BC_DoubleMuParked_Muons.root") as f:
    tree = f["Events"]
    total = tree.num_entries
    writer = None

    for start in range(0, total, BATCH_SIZE):
        stop = min(start + BATCH_SIZE, total)
        print(f"  Batch {start:,} → {stop:,} ...", end=" ")

        # Lecture uproot → awkward → PyArrow directement
        # Pas de Pandas, pas de concat — on écrit au fur et à mesure
        arrays = tree.arrays(
            ["nMuon", "Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge"],
            entry_start=start,
            entry_stop=stop,
            library="ak"
        )

        # Filtres appliqués ici directement
        mask = arrays["nMuon"] == 2
        arrays = arrays[mask]

        # Extraction des 2 muons
        pt1  = ak.to_numpy(arrays["Muon_pt"][:, 0])
        pt2  = ak.to_numpy(arrays["Muon_pt"][:, 1])
        eta1 = ak.to_numpy(arrays["Muon_eta"][:, 0])
        eta2 = ak.to_numpy(arrays["Muon_eta"][:, 1])
        phi1 = ak.to_numpy(arrays["Muon_phi"][:, 0])
        phi2 = ak.to_numpy(arrays["Muon_phi"][:, 1])
        q1   = ak.to_numpy(arrays["Muon_charge"][:, 0])
        q2   = ak.to_numpy(arrays["Muon_charge"][:, 1])

        # Filtres : charge opposée + pt + eta
        mask2 = (
            (q1 + q2 == 0) &
            (pt1 >= 3) & (pt1 < 100) &
            (pt2 >= 3) & (pt2 < 100) &
            (np.abs(eta1) <= 2.4) &
            (np.abs(eta2) <= 2.4)
        )

        pt1, pt2   = pt1[mask2], pt2[mask2]
        eta1, eta2 = eta1[mask2], eta2[mask2]
        phi1, phi2 = phi1[mask2], phi2[mask2]

        # Écriture PyArrow directement sur disque
        table = pa.table({
            "pt1": pt1, "pt2": pt2,
            "eta1": eta1, "eta2": eta2,
            "phi1": phi1, "phi2": phi2,
        })

        if writer is None:
            writer = pq.ParquetWriter(
                PARQUET_DIR / "dimuon.parquet",
                table.schema,
                compression="snappy"
            )
        writer.write_table(table)
        print(f"OK — {len(pt1):,} paires gardées")

    if writer:
        writer.close()

print(f"\nTerminé. Fichier : {PARQUET_DIR / 'dimuon.parquet'}")

  Batch 0 → 5,000,000 ... OK — 1,850,247 paires gardées
  Batch 5,000,000 → 10,000,000 ... OK — 1,838,256 paires gardées
  Batch 10,000,000 → 15,000,000 ... OK — 1,844,050 paires gardées
  Batch 15,000,000 → 20,000,000 ... OK — 1,855,865 paires gardées
  Batch 20,000,000 → 25,000,000 ... OK — 1,832,597 paires gardées
  Batch 25,000,000 → 30,000,000 ... OK — 1,950,520 paires gardées
  Batch 30,000,000 → 35,000,000 ... OK — 1,976,435 paires gardées
  Batch 35,000,000 → 40,000,000 ... OK — 2,003,249 paires gardées
  Batch 40,000,000 → 45,000,000 ... OK — 2,022,056 paires gardées
  Batch 45,000,000 → 50,000,000 ... OK — 2,028,452 paires gardées
  Batch 50,000,000 → 55,000,000 ... OK — 1,945,259 paires gardées
  Batch 55,000,000 → 60,000,000 ... OK — 1,988,374 paires gardées
  Batch 60,000,000 → 61,540,413 ... OK — 626,362 paires gardées

Terminé. Fichier : data\processed\dimuon_parquet\dimuon.parquet


In [ ]:
spark = (
    SparkSession.builder
    .appName("CERNSparkViz")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet(r"data\processed\dimuon_parquet\dimuon.parquet")
print(f"Lignes chargées : {df.count():,}")

Spark 4.1.1 démarré
Lignes chargées : 23,761,722


## Ingestion — Choix technique : uproot → PyArrow → Parquet

### Le problème

La première approche tentée : lire les 61.5M événements par batches
via uproot, stocker chaque batch en Pandas, puis tout concaténer.

Le `pd.concat` de 13 batches représente l'assemblage de 61.5M lignes
en mémoire en une seule opération — plus d'une heure sans résultat.

### La solution

Lire chaque batch via uproot, appliquer les filtres immédiatement,
et écrire sur disque en Parquet via PyArrow — sans jamais tout charger en RAM.
```
uproot (batch 5M) → awkward → filtre → PyArrow → écriture Parquet
uproot (batch 5M) → awkward → filtre → PyArrow → écriture Parquet
...  × 13 batches
```

Jamais plus de 5M lignes en mémoire à la fois.
Résultat : **18 secondes** pour 61.5M événements.

## Hypothèse de travail — Spectre de masse invariante

### Contexte

On dispose de 23.7 millions de paires de muons issues de collisions
proton-proton enregistrées par le détecteur CMS au LHC en 2012.

Pour chaque paire, on peut calculer une **masse invariante** —
la masse de la particule qui aurait pu produire ces deux muons
si elle s'était désintégrée en une paire muon+/muon−.

### L'idée centrale

La grande majorité des paires ne provient pas d'une désintégration
spécifique — ce sont deux muons produits indépendamment lors de la collision.
Leur masse invariante est distribuée de façon continue et aléatoire.
C'est ce qu'on appelle le **fond**.

Mais certaines paires proviennent de la désintégration d'une vraie particule.
Toutes ces paires ont la même masse invariante — celle de la particule mère.
En accumulant des millions de paires, ces particules apparaissent comme
des **pics** au-dessus du fond continu.

### Hypothèse

> En calculant la masse invariante de 23.7 millions de paires di-muon
> issues des données CMS 2012, le spectre révèle des pics à des positions
> précises correspondant à des particules connues du Modèle Standard.

### Résultats attendus

| Particule | Masse attendue | Composition | Désintégration | Découverte | Intérêt |
|-----------|---------------|-------------|----------------|------------|---------|
| ρ / ω | ~0.77 GeV | Paire quark-antiquark léger (up/down) | ~100% en paires de pions, rarement en μ+μ− | Années 1960 | Parmi les premières résonances hadroniques observées. Pic faible dans notre dataset car la désintégration en muons est rare. |
| φ | ~1.02 GeV | Paire quark-antiquark étrange (ss̄) | ~15% en μ+μ− | 1964 | Premier méson contenant un quark étrange. Pic modéré attendu. |
| J/ψ | **3.097 GeV** | Paire quark-antiquark charmé (cc̄) | ~6% en μ+μ− | **1974** | **Prix Nobel 1976.** Découverte simultanée à Stanford et Brookhaven — événement majeur de la physique moderne. Pic très net attendu, le plus visible du spectre basse énergie. |
| Υ (upsilon) | **9.460 GeV** | Paire quark-antiquark beauty (bb̄) | ~2.5% en μ+μ− | **1977** | Première particule contenant un quark beauty. Trois pics proches attendus (Υ 1S, 2S, 3S) à 9.46, 10.02 et 10.36 GeV — potentiellement résolubles dans notre spectre. |
| Z⁰ | **91.188 GeV** | Boson médiateur de la force faible | ~3.4% en μ+μ− | **1983** | **Prix Nobel 1984.** Médiateur de la force faible — l'une des quatre forces fondamentales de l'univers. Pic large et très net attendu à 91 GeV, le plus facile à identifier dans notre dataset. |

### Ce que ça démontre

Chaque pic visible est une preuve directe qu'une particule subatomique
existe et se désintègre en paires de muons à cette énergie précise.

### Méthode

1. Calculer la masse invariante pour chaque paire di-muon
2. Construire l'histogramme de la distribution (échelle logarithmique)
3. Identifier visuellement les pics et comparer aux valeurs de référence
4. Annoter chaque pic avec le nom de la particule et sa masse PDG officielle

In [19]:
df = df.withColumn(
    "mass_gev",
    F.sqrt(
        2.0 * F.col("pt1") * F.col("pt2") * (
            F.cosh(F.col("eta1") - F.col("eta2")) -
            F.cos(F.col("phi1") - F.col("phi2"))
        )
    ).cast(FloatType())
).filter(F.col("mass_gev").isNotNull())

# Définir les bins de masse
N_BINS = 1000
MASS_MIN = 0.5
MASS_MAX = 120.0
BIN_WIDTH = (MASS_MAX - MASS_MIN) / N_BINS

# Assigner chaque paire à un bin
df_binned = df.filter(
    (F.col("mass_gev") >= MASS_MIN) &
    (F.col("mass_gev") <= MASS_MAX)
).withColumn(
    "bin",
    F.floor((F.col("mass_gev") - MASS_MIN) / BIN_WIDTH).cast("int")
)

# Compter les paires par bin
df_hist = (
    df_binned
    .groupBy("bin")
    .count()
    .orderBy("bin")
)

# Récupérer en Pandas pour Plotly
hist_pd = df_hist.toPandas()
hist_pd["mass_center"] = MASS_MIN + (hist_pd["bin"] + 0.5) * BIN_WIDTH

In [20]:
# find_peaks cherche les maxima locaux
# prominence : le pic doit dépasser ses voisins d'au moins X
# distance : deux pics doivent être séparés d'au moins X bins
peaks_idx, properties = find_peaks(
    hist_pd["count"],
    prominence=0.04 * hist_pd["count"].max(),  # relative au pic maximum
    distance=3,                                 # au moins 3 bins entre pics                         
)

print(f"Pics détectés automatiquement : {len(peaks_idx)}")
print()
print(f"{'Pic':<6} {'Masse (GeV)':<14} {'Paires au pic':<16} {'Prominence'}")
print("-" * 55)
for i, idx in enumerate(peaks_idx):
    mass  = hist_pd.loc[idx, "mass_center"]
    count = hist_pd.loc[idx, "count"]
    prom  = properties["prominences"][i]
    print(f"{i+1:<6} {mass:<14.3f} {count:<16,} {prom:,.0f}")

Pics détectés automatiquement : 4

Pic    Masse (GeV)    Paires au pic    Prominence
-------------------------------------------------------
1      1.038          279,534          90,037
2      3.069          1,817,907        1,667,962
3      9.403          112,324          91,732
4      90.902         91,100           87,024


In [21]:
print("Comparaison pics détectés vs valeurs PDG officielles :")
print()
print(f"{'Particule':<8} {'Détecté (GeV)':<16} {'PDG (GeV)':<12} {'Écart'}")
print("-" * 50)
comparaison = [
    ("φ",   1.038, 1.020),
    ("J/ψ", 3.069, 3.097),
    ("Υ",   9.403, 9.460),
    ("Z⁰",  90.902, 91.188),
]
for name, detected, pdg in comparaison:
    ecart = abs(detected - pdg) / pdg * 100
    print(f"{name:<8} {detected:<16.3f} {pdg:<12.3f} {ecart:.2f}%")

Comparaison pics détectés vs valeurs PDG officielles :

Particule Détecté (GeV)    PDG (GeV)    Écart
--------------------------------------------------
φ        1.038            1.020        1.76%
J/ψ      3.069            3.097        0.90%
Υ        9.403            9.460        0.60%
Z⁰       90.902           91.188       0.31%


## Résultats — Spectre de masse invariante

### Méthode de détection des pics

L'algorithme `find_peaks` (scipy) détecte automatiquement les maxima locaux
du spectre en imposant deux contraintes :
- **Prominence** : le pic doit dépasser son fond local d'au moins 50 000 paires
- **Distance** : deux pics doivent être séparés d'au moins 10 bins (~1.2 GeV)

### Pics détectés

| Pic | Masse détectée | Paires au pic | Prominence |
|-----|---------------|---------------|------------|
| 1 | 1.038 GeV | 279 534 | 90 037 |
| 2 | 3.069 GeV | 1 817 907 | 1 667 962 |
| 3 | 9.403 GeV | 112 324 | 91 732 |
| 4 | 90.902 GeV | 91 100 | 87 024 |

**Prominence** : mesure à quel point un pic se détache de son fond local.
Une prominence élevée indique un pic net et bien défini.

### Comparaison avec les valeurs officielles (PDG)

| Particule | Masse détectée | Masse PDG | Écart |
|-----------|---------------|-----------|-------|
| φ | 1.038 GeV | 1.020 GeV | 1.76% |
| J/ψ | 3.069 GeV | 3.097 GeV | 0.90% |
| Υ | 9.403 GeV | 9.460 GeV | 0.60% |
| Z⁰ | 90.902 GeV | 91.188 GeV | 0.31% |

### Ce qu'on observe

**4 particules identifiées automatiquement** à partir de 22.8 millions
de paires di-muon.

**Les écarts diminuent avec la masse** : 1.76% pour le φ, 0.31% pour le Z⁰.

**Le ρ/ω à 0.77 GeV n'est pas détecté comme pic distinct** — il est
probablement noyé dans le fond continu à basse masse, ou fusionné
avec le φ voisin compte tenu de la largeur de nos bins (0.119 GeV).

**Le J/ψ est le signal dominant** avec 1.8M paires au pic et une prominence
de 1.67M. C'est la résonance la plus proprement reconstruite dans ce dataset.

In [23]:
## On plot le spectre de masse

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hist_pd["mass_center"],
    y=hist_pd["count"],
    mode="lines",
    fill="tozeroy",
    name="Paires di-muon",
    line=dict(color="steelblue", width=1),
))

## On ajoute les annotations

pics_detectes = [
    (1.038,  "φ",   "1.038 GeV"),
    (3.069,  "J/ψ", "3.069 GeV"),
    (9.403,  "Υ",   "9.403 GeV"),
    (90.902, "Z⁰",  "90.902 GeV"),
]

for mass, name, label in pics_detectes:
    fig.add_vline(
        x=mass,
        line_dash="dash",
        line_color="tomato",
        line_width=1.5,
    )
    fig.add_annotation(
        x=np.log10(mass),
        y=1,
        yref="paper",
        text=f"{name}<br>{label}",
        showarrow=False,
        textangle=-90,
        font=dict(color="tomato", size=11),
        yanchor="top",
        xanchor="left",
    )

# Graduations explicites, sinon on ne voyait rien
tickvals = [0.5, 1, 2, 3, 5, 10, 20, 50, 91, 120]

fig.update_layout(
    title="Spectre de masse invariante di-muon — CMS Open Data 2012",
    xaxis=dict(
        title="Masse invariante (GeV)",
        type="log",
        tickvals=tickvals,
        ticktext=[str(v) for v in tickvals],
        showgrid=True,
    ),
    yaxis=dict(
        title="Nombre de paires",
        type="log", # On passe en base log pour mieux voir (sinon les premières valeurs se superposaient)
    ),
    template="plotly_dark",
    height=550,
)
fig.show()

## En base log l'espace des x évolue en faisant x10
## La même distance entre 0.5 et 5 et la même qu'entre 5 et 50, etc

## Observations — Spectre de masse invariante

### Ce qu'on voit

En traçant le spectre de masse invariante sur 22.8 millions de paires di-muon,
plusieurs choses attirent l'attention.

**Les pics attendus sont là** — à 1, 3, 9 et 91 GeV, on observe bien
des pics qui correspondent aux particules qu'on cherchait.

**Mais certaines choses sont surprenantes.**

---

### Potentiels points d'intérêt

**Pourquoi on ne voit pas le ρ/ω à 0.77 GeV ?**
On s'attendait à un pic à cette masse, mais le spectre est plat dans cette zone.
Nos bins sont probablement trop larges pour le voir correctement
Possible aussi que la particule se désintègre rarement en muon
En tout cas, ce pic semble caché dans le bruit de fond

**Ces petits pics après J/ψ et φ**
Juste après chaque grand pic, on voit un pic plus petit.
Après vérification, ce sont des particules dans des états excités,
des versions plus lourdes de la même famille.
Le petit pic après J/ψ correspond au ψ(2S) à 3.686 GeV, par exemple.

**La bosse vers 25 GeV**
Il y a une élévation du spectre autour de 25 GeV qui ne correspond
à aucune des résonances qu'on cherchait.
Ce n'est pas un pic net — plutôt une bosse diffuse.
Il s'avère que c'est du "fond" lié à la production de quarks lourds.

**Pourquoi le spectre touche un minimum juste avant le Z⁰ ?**
Entre 70 et 80 GeV, le nombre de paires est au plus bas,
puis remonte brutalement pour former le pic Z⁰ à 91 GeV.
C'est contre-intuitif — on aurait pu s'attendre à une montée progressive.
En réalité c'est la signature d'un pic très étroit et très net :
le Z⁰ domine tellement sa région qu'il crée ce contraste avec le fond.

**Pourquoi plus rien après 91 GeV ?**
La descente est très abrupte après le Z⁰.
Est-ce que les particules s'arrêtent de se produire à cette énergie ?

---

### Ce que ça nous apprend sur nos choix techniques

**Nos bins sont peut-être trop grossiers.**
Avec 1000 bins sur 0-120 GeV, chaque bin fait 0.119 GeV.
C'est suffisant pour voir le Z⁰ qui est large,
mais probablement trop grossier pour résoudre les pics étroits à basse masse.
Une prochaine étape naturelle serait de zoomer sur des régions précises
avec des bins plus fins — par exemple uniquement 0.5-5 GeV
pour mieux voir φ et J/ψ.

---

## Prochaine étape — La géométrie du détecteur

Lors de l'analyse des variables, on avait noté quelque chose :
les deux "bouchons" du détecteur (endcaps) ne sont pas symétriques.

- Endcap gauche : 150 307 muons
- Endcap droit  : 160 079 muons
- Écart         : ~6%

> Est-ce que cet écart, et d'autres asymétries éventuelles,
> sont liées à la nature des interactions ou à l'accélérateur ?

In [24]:
# Récupérer eta et phi depuis le DataFrame Spark en Pandas
df_geo = df.select("eta1", "eta2", "phi1", "phi2").toPandas()

eta_all = np.concatenate([df_geo["eta1"].values, df_geo["eta2"].values])
phi_all = np.concatenate([df_geo["phi1"].values, df_geo["phi2"].values])

In [25]:
# 100x100 bins — assez fin pour voir les structures du détecteur
h, eta_edges, phi_edges = np.histogram2d(
    eta_all, phi_all,
    bins=[100, 100],
    range=[[-2.4, 2.4], [-np.pi, np.pi]]
)

# Centre de chaque bin
eta_centers = (eta_edges[:-1] + eta_edges[1:]) / 2
phi_centers = (phi_edges[:-1] + phi_edges[1:]) / 2

print(f"Bin le plus peuplé : {h.max():,.0f} muons")
print(f"Bin le moins peuplé : {h.min():,.0f} muons")
print(f"Moyenne par bin : {h.mean():,.0f} muons")

Bin le plus peuplé : 6,927 muons
Bin le moins peuplé : 1,144 muons
Moyenne par bin : 4,752 muons


In [26]:
fig = go.Figure(go.Heatmap(
    x=eta_centers,
    y=phi_centers,
    z=h.T,
    colorscale="Plasma",
    colorbar=dict(title="Muons"),
))

fig.update_layout(
    title="Géométrie du détecteur CMS — Distribution eta/phi des muons",
    xaxis_title="eta (position angulaire)",
    yaxis_title="phi (angle azimutal, radians)",
    template="plotly_dark",
    height=500,
)

fig.show()

### Observations

## Endcaps

**L'endcap négatif (eta  € [-2.4, -1.5]) semble plus froid que l'endcap positif (eta  € [1.5, -2.4])**

## Barrel central

**On observe deux points de froid aux alentours de (-0.25, 1.1) et (0.25, 1.6)**

## Objectifs

> Déterminer précisément les zones de froid du barrel central
> et quantifier la potentielle assymétrie des endcaps

In [ ]:
h_df = pd.DataFrame(h.T, index=phi_centers, columns=eta_centers) # Notre grille transformée en df
all_bins = h_df.stack().reset_index()
all_bins.columns = ["phi", "eta", "count"]
all_bins = all_bins.sort_values("count")

print(f"\nMoyenne globale : {all_bins['count'].mean():,.0f} muons/bin")
print("\n")

# Quantifier l'asymétrie endcap
barrel_neg = h[:25, :].mean()   # premier quart -> correspond à eta -2.4 à -1.5 
barrel_pos = h[75:, :].mean()   # dernuer quart -> correspond à eta +1.5 à +2.4
barrel_cen = h[25:75, :].mean() # espace entre les deux -> correspond à eta -1.5 à +1.5

print(f"  Endcap négatif (eta < -1.5) : {barrel_neg:,.0f} muons/bin")
print(f"  Barrel central              : {barrel_cen:,.0f} muons/bin")
print(f"  Endcap positif (eta > +1.5) : {barrel_pos:,.0f} muons/bin")
print(f"\nAsymétrie endcaps : {abs(barrel_pos-barrel_neg)/barrel_neg*100:.1f}%") # Ratio moyenne des muons par bin

# Localiser les zones froides locales

# 5 bins les plus chauds et 5 les plus froids


print("5 zones les plus FROIDES :")
print(all_bins.head(5).to_string(index=False))

print("\n5 zones les plus CHAUDES :")
print(all_bins.tail(5).to_string(index=False))




Moyenne globale : 4,752 muons/bin


  Endcap négatif (eta < -1.5) : 3,424 muons/bin
  Barrel central              : 6,051 muons/bin
  Endcap positif (eta > +1.5) : 3,484 muons/bin

Asymétrie endcaps : 1.8%
5 zones les plus FROIDES :
      phi    eta  count
 0.219911  2.376 1144.0
 0.157080  2.376 1151.0
 0.282743 -1.704 1234.0
-2.481858 -2.376 1241.0
 0.157080 -2.376 1287.0

5 zones les plus CHAUDES :
      phi   eta  count
-1.288053  0.36 6881.0
-1.225221  0.36 6890.0
 1.916372 -0.36 6907.0
-2.796017  0.36 6916.0
 1.665044 -0.36 6927.0


## Observations 

Il existe bien une différence relative entre les deux encaps
Seulement **1.8%** après filtrage des données
Alors que près de **6%** avant filtrage
> Il serait intéressant de comprendre la raison de cette asymétrie relative

Sans surprise, les zones les **plus froides** se trouvent
à proximité des **Endcaps**
Alors que les **plus chaudes**, se situent proches du **Barrel**
> Il nous reste à expliquer les zones froides trouvées dans le Barrel

In [ ]:
# Chercher uniquement dans le barrel central
mask_barrel = (np.abs(eta_centers) < 1.4)
h_barrel = h[mask_barrel, :]

# Zones froides dans le barrel uniquement
h_barrel_df = pd.DataFrame(
    h_barrel.T,
    index=phi_centers,
    columns=eta_centers[mask_barrel]
)

threshold_barrel = h_barrel.mean() * 0.75  # 85% de la moyenne barrel
cold_barrel = h_barrel_df[h_barrel_df < threshold_barrel].stack().reset_index()
cold_barrel.columns = ["phi", "eta", "count"]
cold_barrel = cold_barrel.sort_values("count")

print(f"Moyenne barrel : {h_barrel.mean():,.0f} muons/bin")
print(f"Seuil anomalie : {threshold_barrel:,.0f} muons/bin")
print(f"Zones froides dans le barrel : {len(cold_barrel)} bins")
print()
print(cold_barrel.head(5).to_string(index=False))

Moyenne barrel : 5,910 muons/bin
Seuil anomalie : 4,433 muons/bin
Zones froides dans le barrel : 5800 bins

     phi   eta  count
1.602212 0.264 2639.0
1.539380 0.264 3052.0
1.665044 0.264 3083.0
1.476549 0.264 3213.0
1.727876 0.264 3532.0


## Observations

On peut clairement observer des **points de froid**
dans le Barrel.
Le seuil d'anomalie étant volontairement **en-dessous de la moyenne** (4433 vs 4752 au global. Et 5910 dans le barrel)

> Il serait donc maintenant intéressant de définir nos intervals de froid

In [ ]:
# Créer une grille binaire : 1 = bin froid, 0 = normal
# Sur le barrel uniquement
h_barrel_grid = h_barrel.copy()
cold_mask = h_barrel_grid < 4433 # Notre seuil d'anomalie

# label() identifie automatiquement les groupes de bins adjacents
labeled, n_clusters = label(cold_mask)

print(f"Clusters détectés : {n_clusters}")
print()

for i in range(1, n_clusters + 1):
    cluster_idx = np.where(labeled == i)
    eta_vals = eta_centers[mask_barrel][cluster_idx[0]]
    phi_vals = phi_centers[cluster_idx[1]]
    counts   = h_barrel_grid[cluster_idx]
    
    print(f"Cluster {i}")
    print(f"  eta  : {eta_vals.min():.3f} à {eta_vals.max():.3f}")
    print(f"  phi  : {phi_vals.min():.3f} à {phi_vals.max():.3f} ")
    print(f"  bins : {len(counts)}")
    print(f"  densité moyenne  : {counts.mean():,.0f} muons/bin")

Clusters détectés : 3

Cluster 1
  eta  : -0.264 à -0.264
  phi  : 0.723 à 1.288 
  bins : 10
  densité moyenne  : 3,998 muons/bin
Cluster 2
  eta  : 0.216 à 0.312
  phi  : 1.225 à 1.916 
  bins : 18
  densité moyenne  : 3,813 muons/bin
Cluster 3
  eta  : 1.368 à 1.368
  phi  : 2.293 à 2.293 
  bins : 1
  densité moyenne  : 4,398 muons/bin


: 

## Analyse — Géométrie du détecteur CMS

### Objectif

Lors de l'EDA, on avait noté une asymétrie de ~6% entre les deux endcaps.
L'idée ici : utiliser la distribution eta/phi des 47.5M muons
pour "photographier" le détecteur à partir de ses propres données.

### Résultats globaux

| Zone | Densité moyenne | 
|------|----------------|
| Endcap négatif (eta < -1.5) | 3 424 muons/bin |
| Barrel central (-1.5 à +1.5) | 6 051 muons/bin |
| Endcap positif (eta > +1.5) | 3 484 muons/bin |
| Asymétrie endcaps | 1.8% |

L'asymétrie tombe à 1.8% sur les paires filtrées contre 6% sur les muons bruts.
Le filtre de charge opposée et de pt a rééquilibré la distribution.

### Zones froides locales — barrel

En cherchant les bins dont la densité est inférieure à 4 433 muons
(définit comme seuil d'anomalie à 75% en-dessous de la moyenne du Barrel),
On a réussi à identifier 3 clusters de bins.

| Cluster | eta | phi | Bins | Densité moy |
|---------|-----|-----|------|-------------|
| 1 | -0.264 | 0.723 à 1.288 | 10 | 3 998 |
| 2 | 0.216 à 0.312 | 1.225 à 1.916 | 18 | 3 813 |
| 3 | 1.368 | 2.293| 1 | 4 398 |

**Cluster 3** : un seul bin isolé — probablement une fluctuation statistique

**Clusters 1 et 2** : quasi-symétriques en eta (±0.264) et proches en phi.
Ce sont des zones froides réelles du barrel 

> Il serait intéressant de comprendre la raison d'existence de ces zones froides dans le LHC, qui ne sont très vraisemblablement pas des fluctuations statistiques